# Assignment 5 — Catching Data Leakage Before It Catches You
**Feature Engineering & MLOps** · Target Leakage & Preprocessing Leakage

This notebook demonstrates both kinds of leakage with real numbers and then fixes them:

- **4.1** Target leakage: prove it, don't just assert it
- **4.2** Preprocessing leakage: the wrong order vs. the right order
- **4.3** The fix: an honest, deployable model
- **Bonus** Cross-validation can leak too

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

pd.set_option("display.precision", 6)
RANDOM_STATE = 42
TEST_SIZE = 0.25

# Dataset location: the path from the assignment, with fallbacks for other setups
CANDIDATES = ["data/raw/customer_churn_a5.csv", "customer_churn_a5.csv",
              "/mnt/user-data/uploads/customer_churn_a5.csv"]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))
print("Reading:", DATA_PATH)

Reading: /mnt/user-data/uploads/customer_churn_a5.csv


# 4.1 — Target leakage: prove it, don't just assert it

### Step 1 — Load the data and check the missing-value pattern

In [2]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,churn,days_since_cancellation,final_bill_amount
0,20406,13,99.17,Month-to-month,2,0,NaN,NaN
1,20531,24,66.77,Month-to-month,1,0,NaN,NaN
2,20307,15,44.03,One year,1,0,NaN,NaN
3,20391,3,61.08,Two year,3,0,NaN,NaN
4,20147,8,106.72,Two year,1,0,NaN,NaN


In [3]:
n_non_null = df["days_since_cancellation"].notna().sum()
n_churn = (df["churn"] == 1).sum()
print(f"Rows with non-null days_since_cancellation : {n_non_null}")
print(f"Rows with churn == 1                       : {n_churn}")

# Single-line proof: the non-null mask IS the churn mask, row for row
print("Non-null pattern exactly equals churn=1 rows:",
      df["days_since_cancellation"].notna().equals(df["churn"].eq(1)))

Rows with non-null days_since_cancellation : 149
Rows with churn == 1                       : 149
Non-null pattern exactly equals churn=1 rows: True


**Finding:** 149 rows have a non-null `days_since_cancellation`, and 149 rows have `churn == 1`. The one-line check confirms the *pattern* matches exactly, row for row, not just the count. The same holds for `final_bill_amount`. These two columns exist only when the customer has already cancelled, so they are consequences of the label rather than predictors of it.

### Step 2 — Correlation with the target

In [4]:
corr_leak = df["final_bill_amount"].fillna(-1).corr(df["churn"])
corr_legit = df["tenure_months"].corr(df["churn"])

print(f"corr(final_bill_amount [NaN -> -1], churn) = {corr_leak:+.4f}")
print(f"corr(tenure_months, churn)                 = {corr_legit:+.4f}")

# For context: every legitimate numeric feature
legit_corr = df[["tenure_months", "monthly_charges", "support_calls"]].corrwith(df["churn"])
print()
print(legit_corr.round(4).to_string())

corr(final_bill_amount [NaN -> -1], churn) = +0.9250
corr(tenure_months, churn)                 = -0.2207

tenure_months     -0.2207
monthly_charges    0.1134
support_calls      0.0783


**Finding:** `final_bill_amount` (missing filled with -1) has a correlation of about **+0.93** with `churn`, while the strongest legitimate feature, `tenure_months`, sits at about **-0.22**. A correlation that high from a single column in a noisy behavioural problem like churn is a red flag. The reason is mechanical: after `fillna(-1)`, the column is effectively an "is this a cancelled account?" indicator.

### Steps 3 & 4 — Model A (legitimate features) vs. Model B (with leaked features)

In [5]:
LEGIT_NUM = ["tenure_months", "monthly_charges", "support_calls"]
LEGIT_CAT = ["contract_type"]
LEAKED = ["days_since_cancellation", "final_bill_amount"]
NUMERIC_ALL = LEGIT_NUM + LEAKED

y = df["churn"]

def evaluate(X, y, label):
    """Split first, fit preprocessing on TRAIN only, report test accuracy and ROC-AUC."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    X_tr = pd.get_dummies(X_tr, columns=LEGIT_CAT, dtype=float)
    X_te = pd.get_dummies(X_te, columns=LEGIT_CAT, dtype=float).reindex(columns=X_tr.columns, fill_value=0.0)
    num_cols = [c for c in X_tr.columns if c in NUMERIC_ALL]       # scale numeric columns only
    scaler = StandardScaler().fit(X_tr[num_cols])                   # train only
    def prep(X):
        X = X.copy(); X[num_cols] = scaler.transform(X[num_cols]); return X
    clf = LogisticRegression(max_iter=1000).fit(prep(X_tr), y_tr)
    proba = clf.predict_proba(prep(X_te))[:, 1]
    pred = clf.predict(prep(X_te))
    res = {"model": label, "accuracy": accuracy_score(y_te, pred), "roc_auc": roc_auc_score(y_te, proba)}
    return res, clf, X_tr.columns

# Model A: legitimate features only
res_A, clf_A, cols_A = evaluate(df[LEGIT_NUM + LEGIT_CAT], y, "Model A (legit only)")

# Model B: legit + leaked features (missing -> -1)
X_B = df[LEGIT_NUM + LEGIT_CAT + LEAKED].fillna(-1)
res_B, clf_B, cols_B = evaluate(X_B, y, "Model B (legit + leaked)")

results = pd.DataFrame([res_A, res_B]).set_index("model")
results

,accuracy,roc_auc
model,,
Model A (legit only),0.753333,0.778283
Model B (legit + leaked),1.000000,1.000000


### Step 5 — The gap

In [6]:
gap_acc = res_B["accuracy"] - res_A["accuracy"]
gap_auc = res_B["roc_auc"] - res_A["roc_auc"]
print(f"Accuracy gap (B - A) : {gap_acc:+.4f}")
print(f"ROC-AUC gap  (B - A) : {gap_auc:+.4f}")

# Which features does Model B lean on?
coefs = pd.Series(clf_B.coef_[0], index=cols_B).sort_values(key=abs, ascending=False)
print("\nModel B standardized coefficients (largest magnitude first):")
print(coefs.round(3).to_string())

Accuracy gap (B - A) : +0.2467
ROC-AUC gap  (B - A) : +0.2217

Model B standardized coefficients (largest magnitude first):
final_bill_amount               3.603
days_since_cancellation         1.753
monthly_charges                -0.978
contract_type_Month-to-month    0.663
contract_type_One year         -0.353
contract_type_Two year         -0.323
tenure_months                  -0.308
support_calls                   0.127


**Gap:** Model A (honest) scores about **0.75 accuracy / 0.78 ROC-AUC**. Model B scores a perfect **1.00 / 1.00**, a jump of roughly +0.25 accuracy and +0.22 AUC. Model B's largest coefficients are the two leaked columns.

**What happens in production:** A brand-new applicant has not cancelled anything yet, so `days_since_cancellation` and `final_bill_amount` do not exist for them. The pipeline would either have no value to give (so they get filled with -1) or be fed junk. Model B learned that "-1" means "stayed", so it would predict *no churn* for essentially every applicant, including the ones who will leave. The 100% test score came from reading the answer key, so real accuracy would collapse towards the base rate (about 75% are non-churners) while the model catches almost none of the actual churners.

# 4.2 — Preprocessing leakage: the wrong order vs. the right order

Only the legitimate **numeric** features are used here: `tenure_months`, `monthly_charges`, `support_calls`.

### Step 1 — WRONG order: fit the scaler on the entire dataset, then split

In [7]:
X_num = df[LEGIT_NUM]

scaler_wrong = StandardScaler().fit(X_num)               # sees ALL 600 rows, including future test rows
X_scaled_wrong = scaler_wrong.transform(X_num)
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    X_scaled_wrong, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print("WRONG (fit on full dataset)")
print("mean_ :", scaler_wrong.mean_)
print("scale_:", scaler_wrong.scale_)

WRONG (fit on full dataset)
mean_ : [19.56166667 65.92201667  2.165     ]
scale_: [17.52454081 22.79931919  1.49814608]


### Step 2 — RIGHT order: split first, fit a new scaler on the training split only

In [8]:
Xtr_raw, Xte_raw, ytr, yte = train_test_split(
    X_num, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

scaler_right = StandardScaler().fit(Xtr_raw)             # training rows only
print("RIGHT (fit on training split only)")
print("mean_ :", scaler_right.mean_)
print("scale_:", scaler_right.scale_)

RIGHT (fit on training split only)
mean_ : [19.89555556 65.79473333  2.11777778]
scale_: [17.44515795 22.26390441  1.48380725]


### Step 3 — Numeric difference between the two scalers

In [9]:
comparison = pd.DataFrame({
    "mean_wrong": scaler_wrong.mean_,  "mean_right": scaler_right.mean_,
    "mean_diff (wrong - right)": scaler_wrong.mean_ - scaler_right.mean_,
    "scale_wrong": scaler_wrong.scale_, "scale_right": scaler_right.scale_,
    "scale_diff (wrong - right)": scaler_wrong.scale_ - scaler_right.scale_,
}, index=LEGIT_NUM)
comparison

,mean_wrong,mean_right,mean_diff (wrong - right),scale_wrong,scale_right,scale_diff (wrong - right)
tenure_months,19.561667,19.895556,-0.333889,17.524541,17.445158,0.079383
monthly_charges,65.922017,65.794733,0.127283,22.799319,22.263904,0.535415
support_calls,2.165000,2.117778,0.047222,1.498146,1.483807,0.014339


**Difference:** The means differ by about -0.33 (`tenure_months`), +0.13 (`monthly_charges`) and +0.05 (`support_calls`); the scales differ by up to about 0.54 (`monthly_charges`). The gap is small on this dataset, but the principle does not depend on its size. The test set is meant to stand in for data the model has never seen. If the scaler is fit on all rows, the test rows have already influenced the preprocessing the model is trained with, so the evaluation is no longer a clean simulation of deployment. On a small dataset, with outliers, or with a more aggressive transform (PCA, target encoding, feature selection), the same mistake can inflate scores a lot. The correct order is cheap and always safe.

### Step 4 — Rebuild with a single `sklearn.Pipeline` (imputer + scaler + model)

In [10]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   LogisticRegression(max_iter=1000)),
])
pipe.fit(Xtr_raw, ytr)                                   # fit on the training split only

pipe_scaler = pipe.named_steps["scaler"]
print("Pipeline scaler mean_ :", pipe_scaler.mean_)
print("Pipeline scaler scale_:", pipe_scaler.scale_)

same_mean = np.allclose(pipe_scaler.mean_, scaler_right.mean_)
same_scale = np.allclose(pipe_scaler.scale_, scaler_right.scale_)
print("\nMatches manual 'right' mean_ :", same_mean)
print("Matches manual 'right' scale_:", same_scale)
assert same_mean and same_scale

Pipeline scaler mean_ : [19.89555556 65.79473333  2.11777778]
Pipeline scaler scale_: [17.44515795 22.26390441  1.48380725]

Matches manual 'right' mean_ : True
Matches manual 'right' scale_: True


**Result:** The scaler inside the `Pipeline`, fit on the training split only, produces exactly the same `mean_` and `scale_` as the manual "correct" version. The assertion passes. The pipeline also makes the right order automatic: `fit` only ever sees training data, and `predict` reuses the learned statistics.

# 4.3 — The fix: legitimate features + properly ordered pipeline

In [11]:
X_final = df[LEGIT_NUM + LEGIT_CAT]
Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(
    X_final, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)   # 1) split FIRST

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), LEGIT_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), LEGIT_CAT),
])
final_model = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])
final_model.fit(Xtr_f, ytr_f)                                                # 2) fit on TRAIN only

final_pred = final_model.predict(Xte_f)
final_proba = final_model.predict_proba(Xte_f)[:, 1]
res_final = {"accuracy": accuracy_score(yte_f, final_pred), "roc_auc": roc_auc_score(yte_f, final_proba)}

summary = pd.DataFrame({"Model A (4.1)": {"accuracy": res_A["accuracy"], "roc_auc": res_A["roc_auc"]},
                        "Final pipeline (4.3)": res_final}).T
display(summary)

match_acc = np.isclose(res_final["accuracy"], res_A["accuracy"])
match_auc = np.isclose(res_final["roc_auc"], res_A["roc_auc"])
print("Accuracy matches Model A:", match_acc)
print("ROC-AUC matches Model A :", match_auc)
assert match_acc and match_auc

,accuracy,roc_auc
Model A (4.1),0.753333,0.778283
Final pipeline (4.3),0.753333,0.778283


Accuracy matches Model A: True
ROC-AUC matches Model A : True


**Result:** Using only the legitimate features, splitting first and fitting all preprocessing (imputer, scaler, one-hot encoder) on the training split, the final model reproduces Model A's test accuracy and ROC-AUC exactly (about **0.753 / 0.778**). This is the honest, deployable model: its score is what we can realistically expect on new customers.

# Bonus — Cross-validation can leak too

In [12]:
X_cv = df[LEGIT_NUM]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# LEAKY: scaler fit on the whole dataset BEFORE the CV split
X_cv_scaled = StandardScaler().fit_transform(X_cv)
leaky_acc = cross_val_score(LogisticRegression(max_iter=1000), X_cv_scaled, y, cv=cv, scoring="accuracy")
leaky_auc = cross_val_score(LogisticRegression(max_iter=1000), X_cv_scaled, y, cv=cv, scoring="roc_auc")

# CORRECT: scaler lives INSIDE the pipeline, so it is re-fit on each fold's training part
safe_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
safe_acc = cross_val_score(safe_pipe, X_cv, y, cv=cv, scoring="accuracy")
safe_auc = cross_val_score(safe_pipe, X_cv, y, cv=cv, scoring="roc_auc")

cv_table = pd.DataFrame({
    "leaky mean": [leaky_acc.mean(), leaky_auc.mean()],
    "leaky std":  [leaky_acc.std(),  leaky_auc.std()],
    "pipeline mean": [safe_acc.mean(), safe_auc.mean()],
    "pipeline std":  [safe_acc.std(),  safe_auc.std()],
}, index=["accuracy", "roc_auc"])
cv_table["mean diff (leaky - pipeline)"] = cv_table["leaky mean"] - cv_table["pipeline mean"]
cv_table

,leaky mean,leaky std,pipeline mean,pipeline std,mean diff (leaky - pipeline)
accuracy,0.746667,0.015456,0.746667,0.015456,0.000000
roc_auc,0.672790,0.036594,0.672641,0.036687,0.000148


**Finding:** The leaky CV and the pipeline CV are nearly identical here: accuracy is the same (about 0.747) and ROC-AUC differs by only about 0.0001 (0.6728 vs 0.6726).

**Why the difference is tiny on this dataset:** A `StandardScaler` only learns a mean and a standard deviation per column. With 600 rows and 3 numeric features, each fold's statistics are almost identical to the global ones, and logistic regression just rescales its coefficients for a shifted and stretched feature. So there is almost no information about the validation folds for the model to exploit.

**Why the risk is still real:** The scaler is the mildest preprocessing step. With a small dataset, heavy outliers, PCA, feature selection against the target, target or frequency encoding, or imputation based on group statistics, the same mistake leaks real information and CV scores can be inflated by a lot. Putting every fitted step inside the `Pipeline` that is passed to `cross_val_score` removes the risk at no cost.